In [1]:
## steps:

'''
1. load dependencies
2. load EDM, lava dome , and seismic data
3. plot results
'''

'\n1. load dependencies\n2. load EDM, lava dome , and seismic data\n3. plot results\n'

In [1]:
## first time series test
# append ../../ to make link to main repo work
import sys
sys.path.append('../../')
sys.path.append('/Applications/anaconda3/envs/PhD_clean/lib/python3.11/site-packages/')

# import external packages
import numpy as np
import pandas as pd
import numba
from numba import vectorize
import glob # for file search
import copy
import os # operating system stuff
import re # regex


import geopandas as gpd
import rasterio as rio
import rasterio.mask
from rasterio.plot import plotting_extent
from shapely.geometry import Polygon
from shapely.geometry.point import Point
import pyproj
from pyproj import CRS
from inpoly import inpoly2 # for fast inpolygon checks
import utm

import matplotlib.pyplot as plt 
import matplotlib.dates as mdates
from matplotlib import cm as mpl_cm
from matplotlib import colors as mcolors 
import matplotlib.image as mplimg

from mpl_toolkits.axes_grid1 import make_axes_locatable # for colorbar scaling
from mpl_toolkits.axes_grid1 import ImageGrid
from matplotlib_scalebar.scalebar import ScaleBar
from matplotlib.gridspec import GridSpec
from matplotlib.ticker import FormatStrFormatter

import seaborn as sns
from matplotlib import rc_file_defaults
rc_file_defaults()
# sns.set(style=None, color_codes=True)

from shapely.geometry import Polygon
from shapely.geometry.point import Point
import datetime

import configparser

from cmcrameri import cm # for scientific colourmaps

###########################
# import main local package
import SPOTSAR_main as sm


In [2]:
# load EDM data

# EDM data
import datetime
dates = ['2020-04-24',
         '2020-05-01',
         '2020-05-08',
         '2020-05-15',
         '2020-05-22',
         '2020-05-29',
         '2020-06-05',
         '2020-06-12',
         '2020-06-19',
         '2020-06-26',
         '2020-07-03',
         '2020-07-10',
         '2020-07-17',
         '2020-07-24',
         '2020-07-31',
         '2020-08-07',
         '2020-08-14',
         '2020-08-21',
         '2020-08-28',
         '2020-09-04',
         '2020-09-11',
         '2020-09-18',
         '2020-09-25',
         '2020-10-02',
         '2020-10-09',
         '2020-10-16',
         '2020-10-23',
         '2020-10-30',
         '2020-11-06',
         '2020-11-13',
         '2020-11-20',
         '2020-11-27',
         '2020-12-04',
         '2020-12-11',
         '2020-12-18',
         '2020-12-25',
         '2021-01-01',
         '2021-01-08',
         '2021-01-15',
         '2021-01-22',
         '2021-01-29',
         '2021-02-05',
         '2021-02-12',
         '2021-02-19',
         '2021-02-26',
         '2021-03-05',
         '2021-03-12',
         '2021-03-19',
         '2021-03-26',
         '2021-04-02',
         '2021-04-09',
         '2021-04-16',
         '2021-04-23',
         '2021-04-30',
         '2021-05-07',
         '2021-05-14',
         '2021-05-21',
         '2021-06-04',
         '2021-06-11',
         '2021-06-18',
         '2021-06-25',
         '2021-07-02',
         '2021-07-09',
         '2021-07-16',
         '2021-07-23',
         '2021-07-30',
         '2021-08-06',
         '2021-08-13',
         '2021-08-20',
         '2021-08-27',
         '2021-09-03',
         '2021-09-10',
         '2021-09-17',
         '2021-09-24',
         '2021-10-01',
         '2021-10-08',
         '2021-10-15',
         '2021-10-22',
         '2021-10-29',
         '2021-11-05',
         '2021-11-12',
         '2021-11-19',
         '2021-11-26',
         '2021-12-03',
         '2021-12-10',
         '2021-12-17',
         '2021-12-24',
         '2021-12-31']

EDM_dates = [datetime.datetime.strptime(date,'%Y-%m-%d') for date in dates]

EDM_rate = [   0,    0,    0,    0,    0,    0,    0,    0,   -1, -2/7,
            -2/7, -2/7, -1/7,    0, -2/7, -2/7, -1/7, -1/7, -2/7, -2/7,
            -2/7, -2/7, -3/7, -2/7,   -1,   -2,   -4,   -9,  -10,   -9,
             -11,  -11,   -9,   -9,  -11,  -14,  -15,   -6, -0.8, -0.4,
               0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
               0,    0, -0.5, -0.6, -0.7,    0,   -1, -1.8, -0.7,   -2,
            -0.7,   -4,   -8,   -7,  -11,  -12, -7.7, -1.9,    0,    0,
               0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            -0.2, -0.2, -0.7, -0.5, -0.4, -0.2, -0.2, -0.4]

EDM_dist_min_max = np.array([[4044.834, 4044.840, 3859.108, 3859.115],
                             [4044.836, 4044.841, 3859.106, 3859.112],
                             [4044.830, 4044.840, 3859.102, 3859.114],
                             [4044.832, 4044.837, 3859.106, 3859.113],
                             [4044.836, 4044.842, 3859.108, 3859.114],
                             [4044.836, 4044.841, 3859.110, 3859.116],
                             [4044.834, 4044.840, 3859.110, 3859.116],
                             [4044.835, 4044.846, 3859.110, 3859.115],
                             [4044.769, 4044.841, 3859.055, 3859.122],
                             [4044.738, 4044.763, 3859.027, 3859.051],
                             [4044.719, 4044.738, 3859.011, 3859.031],
                             [4044.697, 4044.718, 3858.996, 3859.017],
                             [4044.690, 4044.701, 3858.995, 3859.003],
                             [4044.681, 4044.691, 3858.990, 3858.998],
                             [4044.659, 4044.682, 3858.976, 3858.990],
                             [4044.640, 4044.657, 3858.962, 3858.974],
                             [4044.626, 4044.641, 3858.952, 3858.963],
                             [4044.604, 4044.621, 3858.934, 3858.948],
                             [4044.580, 4044.598, 3858.925, 3858.932],
                             [4044.558, 4044.574, 3858.905, 3858.922],
                             [4044.540, 4044.556, 3858.902, 3858.908],
                             [4044.510, 4044.526, 3858.875, 3858.893],
                             [4044.466, 4044.500, 3858.849, 3858.874],
                             [4044.433, 4044.459, 3858.828, 3858.844],
                             [4044.319, 4044.338, 3858.759, 3858.772],
                             [4044.183, 4044.319, 3858.663, 3858.759],
                             [4043.795, 4044.117, 3858.347, 3858.639],
                             [4043.102, 4043.780, 3857.719, 3858.334],
                             [4042.361, 4043.057, 3856.911, 3857.681],
                             [4041.388, 4042.227, 3856.099, 3856.899],
                             [4040.604, 4041.383, 3855.342, 3856.090],
                             [4039.829, 4040.567, 3854.566, 3855.308],
                             [4039.179, 4039.801, 3853.850, 3854.537],
                             [4038.727, 4039.087, 3853.442, 3853.816],
                             [4037.761, 4038.238, 3852.457, 3852.954],
                             [4036.760, 4037.753, 3851.429, 3852.448],
                             [4035.710, 4036.748, 3850.333, 3851.419],
                             [4035.318, 4035.707, 3849.916, 3850.328],
                             [4035.243, 4035.297, 3849.843, 3850.899],
                             [4035.220, 4035.245, 3849.812, 3849.837],
                             [4035.213, 4035.218, 3849.804, 3849.811],
                             [4035.164, 4035.214, 3849.763, 3849.802],
                             [4035.158, 4035.168, 3849.748, 3849.763],
                             [4035.148, 4035.160, 3849.741, 3849.758],
                             [4035.146, 4035.156, 3849.742, 3849.753],
                             [4035.145, 4035.153, 3849.738, 3849.749],
                             [4035.131, 4035.148, 3849.729, 3849.746],
                             [4035.124, 4035.137, 3849.724, 3849.735],
                             [4035.093, 4035.124, 3849.705, 3849.725],
                             [4035.091, 4035.098, 3849.700, 3849.709],
                             [4035.083, 4035.097, 3849.695, 3849.708],
                             [4035.059, 4035.090, 3849.678, 3849.702],
                             [4035.021, 4035.058, 3849.651, 3849.681],
                             [4034.976, 4035.022, 3849.616, 3849.653],
                             [4034.910, 4034.973, 3849.557, 3849.616],
                             [4034.885, 4034.903, 3849.539, 3849.554],
                             [4034.820, 4034.887, 3849.485, 3849.542],
                             [4034.623, 4034.742, 3849.311, 3849.421],
                             [4033.827, 4033.874, 3848.518, 3848.558],
                             [4034.441, 4034.572, 3849.143, 3849.267],
                             [4034.403, 4034.452, 3849.105, 3849.154],
                             [4034.084, 4034.385, 3848.797, 3849.087],
                             [4033.528, 4034.078, 3848.231, 3848.793],
                             [4033.039, 4033.522, 3847.721, 3848.222],
                             [4032.321, 4033.033, 3846.975, 3847.716],
                             [4031.519, 4032.220, 3846.140, 3846.867],
                             [4030.990, 4031.499, 3845.602, 3846.119],
                             [4030.780, 4030.845, 3845.388, 3845.529],
                             [4030.770, 4030.785, 3845.375, 3845.390], 
                             [4030.751, 4030.772, 3845.362, 3845.377],
                             [4030.740, 4030.761, 3845.353, 3845.365],
                             [4030.723, 4030.746, 3845.338, 3845.356],
                             [4030.716, 4030.726, 3845.332, 3845.343],
                             [4030.698, 4030.698, 3845.314, 3845.322],
                             [4030.694, 4030.700, 3845.309, 3845.315],
                             [4030.681, 4030.696, 3845.299, 3845.310],
                             [4030.675, 4030.685, 3845.295, 3845.304],
                             [4030.658, 4030.676, 3845.285, 3845.296],
                             [4030.663, 4030.667, 3845.269, 3845.286],
                             [  np.nan,   np.nan, 3845.265, 3845.276],
                             [4030.632, 4030.643, 3845.243, 3845.258],
                             [4030.627, 4030.633, 3845.226, 3845.242],
                             [4030.603, 4030.618, 3845.226, 3845.242],
                             [4030.570, 4030.583, 3845.133, 3845.158],
                             [4030.537, 4030.553, 3845.097, 3845.116],
                             [4030.520, 4030.536, 3845.080, 3845.095],
                             [4030.511, 4030.526, 3845.075, 3845.085],
                             [4030.483, 4030.511, 3845.057, 3845.079]])



#                    AP VTA , VTB, MP, LF, RF, DG, TT
EQ_data = np.array([[0, 0,   0,   47,  0, 16, 10,  8],
                    [0, 0,   0,   76,  0, 50, 34,  14],
                    [0, 0,   0,   40,  1, 47, 24,  13],
                    [0, 0,   0,   13,  7, 16, 16,  15],
                    [0, 0,   0,   18,  7, 29, 20,  8],
                    [0, 0,   0,   28,  8, 4, 37,  12],
                    [0, 0,   0,   45,  9, 13, 72,  17],
                    [0, 42,   1,   39,  3, 24, 74,  27],
                    [0, 45,   3,   59,  29, 26, 50,  14],
                    [0, 5,   2,   75,  2, 24, 26,  15],
                    [0, 1,   11,   56,  2, 22, 23,  14],
                    [0, 2,   8,   46,  3, 15, 11,  20],
                    [0, 0,   12,   55,  4, 28, 13,  20],
                    [0, 2,   0,   44,  3, 37, 16,  19],
                    [0, 0,   10,  56,  2, 36, 11,  16],
                    [0, 0,   8,  69,  2, 38, 9,  14],
                    [0, 0,   7,  72,  2, 21, 10,  17],
                    [0, 0,   9,  67,  1, 42, 10,  19],
                    [0, 1,   7,  73,  4, 23, 20,  10],
                    [0, 1,   9,  63,  7, 29, 6,  24],
                    [0, 2,   15, 130, 18, 40, 18,  12],
                    [0, 0,   16, 122, 7, 24, 14,  7],
                    [0, 1,   15, 134, 7, 32, 30,  11],
                    [0, 0,   19, 194, 21, 65, 32,  10],
                    [0, 0,   41, 319, 5, 67, 56,  10],
                    [0, 0,   63, 433, 23, 170, 167,  16],
                    [0, 0,   81, 864, 10, 367, 286,  7],
                    [0, 0,   193, 1663, 9, 391, 330,  9],
                    [0, 0,   244, 2189, 9, 385, 403,  6],
                    [0, 0,   262, 1939, 7, 441, 352,  8],
                    [0, 0,   277, 2464, 4, 340, 541,  9],
                    [0, 0,   236, 2128, 3, 289, 330, 11],
                    [0, 0,   232, 1692, 5, 256, 209, 2],
                    [0, 0,   217, 1621, 6, 284, 303, 9],
                    [0, 0,   307, 1587, 3, 250, 324, 7],
                    [0, 0,   501, 2403, 4, 343, 494, 8],
                    [4, 0,   541, 2270, 4, 611, 628, 4],
                    [2, 0,   208, 803, 0, 1056, 172, 16],
                    [19, 0,   27, 176, 0, 836, 30, 8],
                    [71, 0,   6, 105, 0, 1148, 122, 1],
                    [1, 0,   0, 31, 0, 574, 14, 8],
                    [0, 0,   22, 193, 0, 775, 10, 13],
                    [0, 0,   3, 60, 0, 943, 17, 12],
                    [3, 0,   0, 14, 0, 985, 37, 5],
                    [10, 0,   1, 4, 0, 1135, 71, 3],
                    [12, 0,   1, 8, 0, 1164, 64, 2],
                    [3, 0,   2, 33, 1, 1059, 64, 15],
                    [4, 0,   0, 29, 1, 911, 15, 2],
                    [14, 0,   3, 61, 0, 1074, 16, 6],
                    [13, 0,   0, 16, 0, 1042, 8, 6],
                    [6, 0,   0, 21, 0, 1038, 13, 10],
                    [9, 0,   12, 156, 0, 924, 14, 4],
                    [12, 0,   11, 102, 0, 992, 16, 6],
                    [12, 0,   29, 152, 1, 941, 50, 6],
                    [4, 0,   37, 199, 1, 1017, 49, 10],
                    [7, 0,   7, 99, 0, 981, 49, 3],
                    [15, 0,  22, 201, 0, 949, 43, 9],
                    [12, 0,  72, 340, 1, 964, 104, 6],
                    [19, 0,  17, 136, 2, 1158, 108, 6],
                    [22, 0,  59, 362, 0, 1325, 122, 11],
                    [39, 0,  21, 141, 0, 1824, 143, 4],
                    [17, 0,  187, 771, 7, 1461, 120, 6],
                    [1, 0,  267, 1222, 3, 1655, 133, 7],
                    [0, 0,  273, 1067, 7, 1476, 145, 6],
                    [4, 0,  468, 1520, 3, 1048, 146, 10],
                    [7, 0,  505, 2415, 5, 1104, 145, 9],
                    [28, 0,  334, 1296, 10, 2021, 141, 19],
                    [20, 0,  79, 228, 3, 1969, 82, 4],
                    [2, 0,  1, 19, 15, 1973, 100, 5],
                    [6, 0,  1, 68, 62, 1978, 1289, 3],
                    [1, 0,  2, 20, 286, 1491, 561, 4],
                    [0, 0,  2, 16, 303, 1559, 1277, 1],
                    [0, 0,  5, 251, 260, 1357, 895, 4],
                    [0, 0,  0, 1268, 244, 1200, 338, 6],
                    [0, 0,  1, 902, 70, 1068, 127, 5],
                    [0, 0,  1, 1465, 133, 1123, 167, 11],
                    [2, 0,  4, 583, 22, 1363, 110, 7],
                    [0, 0,  2, 46, 1, 1199, 93, 82],
                    [1, 0,  0, 33, 0, 1297, 181, 15],
                    [2, 1,  3, 73, 0, 1256, 220, 12],
                    [1, 0,  2, 128, 0, 1245, 384, 10],
                    [1, 1,  2, 70, 0, 938, 89, 1],
                    [2, 0,  12, 116, 2, 1307, 80, 8],
                    [4, 0,  8, 55, 0, 1020, 25, 10],
                    [3, 0,  24, 128, 0, 1190, 43, 8],
                    [4, 0,  10, 52, 0, 999, 15, 7],
                    [2, 0,  1, 30, 0, 1089, 29, 8]])



dates_lavadomes = ['2020-02-19',
                   '2020-06-13',
                   '2020-07-11',
                   '2020-07-26',
                   '2020-10-29',
                   '2020-11-03',
                   '2020-11-16',
                   '2021-01-04',
                   '2021-01-14',
                   '2021-01-21',  
                   '2021-01-25',
                   '2021-01-28',
                   '2021-02-04',
                   '2021-02-11',
                   '2021-02-17',
                   '2021-02-25',
                   '2021-03-05',
                   '2021-03-11',
                   '2021-03-18',
                   '2021-03-25',
                   '2021-04-01',
                   '2021-04-07',
                   '2021-04-08',
                   '2021-04-15',
                   '2021-04-21',
                   '2021-04-22',
                   '2021-04-28',
                   '2021-05-02',
                   '2021-05-16',
                   '2021-05-20',
                   '2021-05-28',
                   '2021-06-08',
                   '2021-06-17',
                   '2021-06-24',
                   '2021-07-01',
                   '2021-07-08',
                   '2021-07-15',
                   '2021-07-22',
                   '2021-07-28',
                   '2021-08-05',
                   '2021-08-19',
                   '2021-08-26',
                   '2021-09-01',
                   '2021-09-09',
                   '2021-09-16',
                   '2021-09-23',
                   '2021-09-30',
                   '2021-10-07',
                   '2021-10-14',
                   '2021-10-21',
                   '2021-10-28',
                   '2021-11-04',
                   '2021-11-11',
                   '2021-11-18',
                   '2021-11-25',
                   '2021-12-02',
                   '2021-12-09',
                   '2021-12-16',
                   '2021-12-23',
                   '2021-12-30',
]

lavadome_dates = [datetime.datetime.strptime(date,'%Y-%m-%d') for date in dates_lavadomes]


lava_dome_vol_rate =np.array([[291000, np.nan, np.nan, np.nan],
                    [200000, np.nan, np.nan, np.nan],
                    [200000, np.nan, np.nan, np.nan],
                    [200000, np.nan, np.nan, np.nan],
                    [200000, np.nan, np.nan, np.nan],
                    [200000, np.nan, np.nan, np.nan],
                    [200000, np.nan, np.nan, np.nan],
                    [200000, np.nan, 0 , np.nan],
                    [np.nan, np.nan, 46766, 8500],
                    [np.nan, np.nan, 104000,8600],
                    [np.nan, np.nan, 157000, np.nan],
                    [np.nan, np.nan, 62000, np.nan],
                    [np.nan, np.nan, 117400, 12600],
                    [np.nan, np.nan, 295000, 48900],
                    [426000,  10000, 397500, 25200],
                    [np.nan, np.nan, 618700, 13600],
                    [np.nan, np.nan, 711000, 13900],
                    [np.nan, np.nan, 785600, 13500],
                    [950000,  12800, 840000, 12900],
                    [np.nan, np.nan, 949000, 13300],
                    [np.nan, np.nan, 1060000, 13400],
                    [np.nan, np.nan, 1098000, 12800],
                    [1681000, np.nan, np.nan, np.nan],
                    [np.nan, np.nan, 1024800, 12200],
                    [np.nan, np.nan, 1069600, 11900],
                    [1794000, np.nan, np.nan, np.nan],
                    [1794000, np.nan, 1141850, 11900],
                    [1700000, 14000, 1100000, 17000],
                    [np.nan, np.nan, 1160000, 11500],
                    [np.nan, np.nan, 1284000, 11700],
                    [np.nan, np.nan, 1264800, 11600],
                    [2100000, np.nan, 1300000, np.nan],
                    [np.nan, np.nan, 1390000, 11300],
                    [np.nan, np.nan, 1590000, 11400],
                    [np.nan, np.nan, 1680000, 11800],
                    [2741000, np.nan, 1815000, np.nan],
                    [2796000, np.nan, 1830000, np.nan],
                    [2808000, np.nan, 1880000, np.nan],
                    [2817000, np.nan, 1878000, np.nan],
                    [np.nan, np.nan, 1895000, np.nan],
                    [np.nan, np.nan, 1350000, np.nan],
                    [2831000, np.nan, 1400000, np.nan],
                    [2842000, np.nan, 1440000, np.nan],
                    [2854000, np.nan, 1550000, np.nan],
                    [2854000, np.nan, 1550000, np.nan],
                    [2854000, np.nan, 1600000, np.nan],
                    [2854000, np.nan, 1630000, np.nan],
                    [2854000, np.nan, 1679000, np.nan],
                    [2927000, np.nan, 1609000, np.nan],
                    [2927000, np.nan, 1609000, np.nan],
                    [2927000, np.nan, 1609000, np.nan],
                    [2927000, np.nan, 1610000, np.nan],
                    [2927000, np.nan, 1610000, np.nan],
                    [2927000, np.nan, 1610000, np.nan],
                    [2927000, np.nan, 1610000, np.nan],
                    [2948000, np.nan, 1615000, np.nan],
                    [3007000, np.nan, 1629000, np.nan],
                    [3007000, np.nan, 1622000, np.nan],
                    [3007000, np.nan, 1654000, np.nan],
                    [3007000, np.nan, 1630000, np.nan]
                   ])

# June 21, 2020 19000 m^3 eroded from southwest sector of Lava-1997
# Oct. 18, 2020 too foggy to get volume measurement
# Nov. 16, 2020 morphology changes due to collapse of old lava ofr lava 1997, 1998, 1888, and 1954
# Dec. 5, 2020 nonew lava dome
# Dec. 14, 2020 no new lava dome
# On January 4, 2021 at 19.50 WIB, the first incandescent lava avalanche was recorded on a seismogram with an amplitude of 33 mm and a duration of 60 seconds.
# Aug 12, reported that SW lava dome volume is 'relatively fixed'

In [4]:
EQ_types = ['Avalanche','VT-Deep (2.5-5 km)','VT-Shallow (<1.5 km)','Multi-Phase','Low-Frequency','Rock Fall','Explosion','Tectonic']

EQ_cumsum = np.fliplr(np.cumsum(EQ_data,axis=1))
# print(np.fliplr(EQ_cumsum))

weight_counts = {
    "TT":  EQ_cumsum[:,0],
    "DG": EQ_cumsum[:,1],
    "RF": EQ_cumsum[:,2],
    "LF":  EQ_cumsum[:,3],
    "MP":  EQ_cumsum[:,4],
    "VTB":  EQ_cumsum[:,5],
    "VTA":  EQ_cumsum[:,6],
    "AP":  EQ_cumsum[:,7]
}

weight_counts = {
    "Tectonic":  EQ_cumsum[:,0],
    "Explosion": EQ_cumsum[:,1],
    "Rock Fall": EQ_cumsum[:,2],
    "Low-Frequency":  EQ_cumsum[:,3],
    "Multi-Phase":  EQ_cumsum[:,4],
    "VT-shallow >1.5 km":  EQ_cumsum[:,5],
    "VT-deep 2.5-5 km":  EQ_cumsum[:,6],
    "Avalanche":  EQ_cumsum[:,7]
}
width=7

In [48]:
i1 = 0
i2 = 25
rockfall_eq = 24533
date_comp1 = EDM_dates[i1]
date_comp2 = EDM_dates[i2]
total_eq = np.sum(EQ_data[i1:i2-1,:])
print(date_comp1)
print(date_comp2)
print(total_eq)
print(((total_eq-rockfall_eq)/(date_comp2-date_comp1).days))


dist_d1 = np.array([np.mean(np.array(EDM_dist_min_max)[i1,0:2]),np.mean(np.array(EDM_dist_min_max)[i1,2:4])])
dist_d2 = np.array([np.mean(np.array(EDM_dist_min_max)[i2,0:2]),np.mean(np.array(EDM_dist_min_max)[i2,2:4])])

dist_diff = dist_d2[:]-dist_d1[:]
disp_rate_comp = [dist_diff[0]/(date_comp2-date_comp1).days, dist_diff[1]/(date_comp2-date_comp1).days]

print(dist_d1)
print(dist_d2)
print(dist_diff)
print(disp_rate_comp)
print(np.mean(disp_rate_comp))


2020-04-24 00:00:00
2020-10-16 00:00:00
3667
-119.23428571428572
[4044.837  3859.1115]
[4044.251 3858.711]
[-0.586  -0.4005]
[-0.003348571428570202, -0.002288571428569932]
-0.002818571428570067


In [67]:
## plot results
## plot EDM and lava dome data
%matplotlib osx

fig=plt.figure(figsize=(12,6.5))
gs=GridSpec(3,1)
ax2=fig.add_subplot(gs[0,0]) # First row, first column
ax0 = fig.add_subplot(gs[1,0]) # second row, first column
ax3 = fig.add_subplot(gs[2,0]) # second row, first column


ax0.set_ylabel('EDM distance change [m]')
ax0_twin = ax0.twinx()
ax0_twin.set_ylabel('EDM change rate\n[cm/day]', color = 'red') 


print(EDM_dist_min_max.shape)
print(np.shape(EDM_dates))


print(EDM_dates[79])
print(EDM_dates[80])
print(EDM_dist_min_max[79,0])
start_week = 0
for week_id in enumerate(EDM_dates[start_week:-1]):
    print(start_week + week_id[0])
    if not np.isnan(EDM_dist_min_max[start_week + week_id[0],0]):
        ax0.fill_between(x= [EDM_dates[start_week + week_id[0]],EDM_dates[start_week+week_id[0]]+datetime.timedelta(7)],
                    y1 = EDM_dist_min_max[start_week + week_id[0],0]-EDM_dist_min_max[start_week,0],
                    y2 = EDM_dist_min_max[start_week + week_id[0],1]-EDM_dist_min_max[start_week,1],
                    color = 'tab:blue',
                    alpha = 0.3)
        ax0.fill_between(x= [EDM_dates[start_week + week_id[0]],EDM_dates[start_week+week_id[0]]+datetime.timedelta(7)],
                    y1 = EDM_dist_min_max[start_week + week_id[0],2]-EDM_dist_min_max[start_week,2],
                    y2 = EDM_dist_min_max[start_week + week_id[0],3]-EDM_dist_min_max[start_week,3],
                    color = 'tab:orange',
                    alpha = 0.3)

ax0_twin.hlines(y=EDM_rate[:-1], xmin=EDM_dates[:-1], xmax=[x +datetime.timedelta(7) for x in EDM_dates[:-1]],
           color='red', zorder=1)
ax0_twin.vlines(x=EDM_dates[1:], ymin=EDM_rate[:-1], ymax=EDM_rate[1:], color='red',
          linestyle='dashed', zorder=1)

ax0.set_ylim([-15.5,0.2])
ax0_twin.set_ylim([-15.5,0.2])

ax0.tick_params(left=True,
                bottom=True,
                top=True,
                labelbottom=False,
                labeltop=False)

ax2.tick_params(left=True,
                bottom=True,
                top=True,
                labelbottom=False,
                labeltop=False)

ax0_twin.spines['right'].set_color('red')
ax0_twin.yaxis.label.set_color('red')
ax0_twin.tick_params(axis='y', colors='red')

##############################################################################################
##############################################################################################


x_lim_dates = ax0.get_xlim()

# # plot lava dome volume and dome growth rate
# # print(lava_dome_vol_rate.shape())
ax2.plot(lavadome_dates,lava_dome_vol_rate[:,0],color='tab:orange', marker='s',linestyle='-')
ax2.plot(lavadome_dates,lava_dome_vol_rate[:,2],color='tab:orange', marker='x',linestyle='--')

ax2_twin = ax2.twinx()
ax2_twin.plot(lavadome_dates,lava_dome_vol_rate[:,1]/(24*3600),color='tab:blue', marker='s',linestyle='-')
ax2_twin.plot(lavadome_dates,lava_dome_vol_rate[:,3]/(24*3600),color='tab:blue', marker='x',linestyle='--')


ax2.spines['right'].set_color('tab:orange')
ax2.yaxis.label.set_color('tab:orange')
ax2.tick_params(axis='y', colors='tab:orange')

ax2_twin.spines['right'].set_color('tab:blue')
ax2_twin.yaxis.label.set_color('tab:blue')
ax2_twin.tick_params(axis='y', colors='tab:blue')

ax2.set_xlim(x_lim_dates)
ax2.set_ylabel(r'Lava dome volume [m$^3$]')
ax2_twin.set_ylabel(r'Extrusion rate [m$^3$/s]')


ax0.fill_between(x= [],
                     y1 = [],
                     y2 = [],
                     color = 'tab:blue',
                     alpha = 0.5,
                     label = 'EDM range BAD-RB1 \n[x+'+ str(EDM_dist_min_max[start_week,0]) + ' m]')
ax0.fill_between(x= [],
                     y1 = [],
                     y2 = [],
                     color = 'tab:orange',
                     alpha = 0.5,
                     label = 'EDM range BAD-RB2 \n[x+' + str(EDM_dist_min_max[start_week,2]) + ' m]')
ax0.hlines(y=[], xmin=[], xmax=[],
              color='red', zorder=1, label = 'EDM rate change [cm/day]')

ax0.legend(loc='lower left')


ax2.plot([],[],color='tab:orange', marker='s',linestyle='-', label='summit dome volume')
ax2.plot([],[],color='tab:orange', marker='x',linestyle='--', label='SW dome volume')
ax2.plot([],[],color='tab:blue', marker='s',linestyle='-', label='summit dome extrusion rate')
ax2.plot([],[],color='tab:blue', marker='x',linestyle='--', label='SW dome extrusion rate')

ax2.legend(loc='upper left')

ax0.axvline(x=datetime.datetime.strptime('2020-10-13','%Y-%m-%d'),linewidth=1.5,color='black',linestyle='--')
ax0.axvline(x=datetime.datetime.strptime('2021-01-04','%Y-%m-%d'),linewidth=1.5,color='black',linestyle='--')
ax0.axvline(x=datetime.datetime.strptime('2021-07-01','%Y-%m-%d'),linewidth=1.5,color='black',linestyle='--')
ax0.axvline(x=datetime.datetime.strptime('2021-08-14','%Y-%m-%d'),linewidth=1.5,color='black',linestyle='--')

ax2.axvline(x=datetime.datetime.strptime('2020-10-13','%Y-%m-%d'),linewidth=1.5,color='black',linestyle='--')
ax2.axvline(x=datetime.datetime.strptime('2021-01-04','%Y-%m-%d'),linewidth=1.5,color='black',linestyle='--')
ax2.axvline(x=datetime.datetime.strptime('2021-07-01','%Y-%m-%d'),linewidth=1.5,color='black',linestyle='--')
ax2.axvline(x=datetime.datetime.strptime('2021-08-14','%Y-%m-%d'),linewidth=1.5,color='black',linestyle='--')

print(np.shape(EQ_data))
print(np.shape(EDM_dates))
# ax3.hist(np.arange(len(EDM_dates[:-1])),bins=EDM_dates,weights=EQ_data,stacked=True,ec='black')

bottom = np.zeros(87)

for boolean, weight_count in weight_counts.items():
    p = ax3.bar(EDM_dates[:-1], weight_count, width, label=boolean, bottom=bottom,align='edge')
# p = ax3.bar(EDM_dates[:-1], weight_counts, width)
ax3.set_ylabel('Event count [stacked]')
ax3.set_xlim(x_lim_dates)
ax3.legend(loc="upper left")
# fig.savefig('')

ax3.axvline(x=datetime.datetime.strptime('2020-10-13','%Y-%m-%d'),linewidth=1.5,color='black',linestyle='--')
ax3.axvline(x=datetime.datetime.strptime('2021-01-04','%Y-%m-%d'),linewidth=1.5,color='black',linestyle='--')
ax3.axvline(x=datetime.datetime.strptime('2021-07-01','%Y-%m-%d'),linewidth=1.5,color='black',linestyle='--')
ax3.axvline(x=datetime.datetime.strptime('2021-08-14','%Y-%m-%d'),linewidth=1.5,color='black',linestyle='--')

ax3.tick_params(left=True,
                bottom=True,
                top=True,
                labelbottom=True,
                labeltop=False)

plt.show()

(88, 4)
(88,)
2021-11-05 00:00:00
2021-11-12 00:00:00
nan
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
(87, 8)
(88,)


In [102]:
## plot results
## plot EDM and lava dome data
%matplotlib osx

fig=plt.figure(figsize=(12,6.5))
gs=GridSpec(10,1,height_ratios=[15,15,4,4,4,4,4,4,4,4])
ax2=fig.add_subplot(gs[0,0]) # First row, first column
ax0 = fig.add_subplot(gs[1,0]) # second row, first column
ax3 = fig.add_subplot(gs[2,0]) # second row, first column
ax4 = fig.add_subplot(gs[3,0]) # second row, first column
ax5 = fig.add_subplot(gs[4,0]) # second row, first column
ax6 = fig.add_subplot(gs[5,0]) # second row, first column
ax7 = fig.add_subplot(gs[6,0]) # second row, first column
ax8 = fig.add_subplot(gs[7,0]) # second row, first column
ax9 = fig.add_subplot(gs[8,0]) # second row, first column
ax10 = fig.add_subplot(gs[9,0]) # second row, first column



ax0.set_ylabel('EDM distance change [m]')
ax0_twin = ax0.twinx()
ax0_twin.set_ylabel('EDM change rate\n[cm/day]', color = 'red') 


print(EDM_dist_min_max.shape)
print(np.shape(EDM_dates))


print(EDM_dates[79])
print(EDM_dates[80])
print(EDM_dist_min_max[79,0])
start_week = 0
for week_id in enumerate(EDM_dates[start_week:-1]):
    print(start_week + week_id[0])
    if not np.isnan(EDM_dist_min_max[start_week + week_id[0],0]):
        ax0.fill_between(x= [EDM_dates[start_week + week_id[0]],EDM_dates[start_week+week_id[0]]+datetime.timedelta(7)],
                    y1 = EDM_dist_min_max[start_week + week_id[0],0]-EDM_dist_min_max[start_week,0],
                    y2 = EDM_dist_min_max[start_week + week_id[0],1]-EDM_dist_min_max[start_week,1],
                    color = 'tab:blue',
                    alpha = 0.3)
        ax0.fill_between(x= [EDM_dates[start_week + week_id[0]],EDM_dates[start_week+week_id[0]]+datetime.timedelta(7)],
                    y1 = EDM_dist_min_max[start_week + week_id[0],2]-EDM_dist_min_max[start_week,2],
                    y2 = EDM_dist_min_max[start_week + week_id[0],3]-EDM_dist_min_max[start_week,3],
                    color = 'tab:orange',
                    alpha = 0.3)

ax0_twin.hlines(y=EDM_rate[:-1], xmin=EDM_dates[:-1], xmax=[x +datetime.timedelta(7) for x in EDM_dates[:-1]],
           color='red', zorder=1)
ax0_twin.vlines(x=EDM_dates[1:], ymin=EDM_rate[:-1], ymax=EDM_rate[1:], color='red',
          linestyle='dashed', zorder=1)

ax0.set_ylim([-15.5,0.2])
ax0_twin.set_ylim([-15.5,0.2])

ax0.tick_params(left=True,
                bottom=True,
                top=True,
                labelbottom=False,
                labeltop=False)

ax2.tick_params(left=True,
                bottom=True,
                top=True,
                labelbottom=False,
                labeltop=False)

ax0_twin.spines['right'].set_color('red')
ax0_twin.yaxis.label.set_color('red')
ax0_twin.tick_params(axis='y', colors='red')

##############################################################################################
##############################################################################################


x_lim_dates = ax0.get_xlim()

# # plot lava dome volume and dome growth rate
# # print(lava_dome_vol_rate.shape())
ax2.plot(lavadome_dates,lava_dome_vol_rate[:,0],color='tab:orange', marker='s',linestyle='-')
ax2.plot(lavadome_dates,lava_dome_vol_rate[:,2],color='tab:orange', marker='x',linestyle='--')

ax2_twin = ax2.twinx()
ax2_twin.plot(lavadome_dates,lava_dome_vol_rate[:,1]/(24*3600),color='tab:blue', marker='s',linestyle='-')
ax2_twin.plot(lavadome_dates,lava_dome_vol_rate[:,3]/(24*3600),color='tab:blue', marker='x',linestyle='--')


ax2.spines['right'].set_color('tab:orange')
ax2.yaxis.label.set_color('tab:orange')
ax2.tick_params(axis='y', colors='tab:orange')

ax2_twin.spines['right'].set_color('tab:blue')
ax2_twin.yaxis.label.set_color('tab:blue')
ax2_twin.tick_params(axis='y', colors='tab:blue')

ax2.set_xlim(x_lim_dates)
ax2.set_ylabel(r'Lava dome volume [m$^3$]')
ax2_twin.set_ylabel(r'Extrusion rate [m$^3$/s]')


ax0.fill_between(x= [],
                     y1 = [],
                     y2 = [],
                     color = 'tab:blue',
                     alpha = 0.5,
                     label = 'EDM range BAD-RB1 \n[x+'+ str(EDM_dist_min_max[start_week,0]) + ' m]')
ax0.fill_between(x= [],
                     y1 = [],
                     y2 = [],
                     color = 'tab:orange',
                     alpha = 0.5,
                     label = 'EDM range BAD-RB2 \n[x+' + str(EDM_dist_min_max[start_week,2]) + ' m]')
ax0.hlines(y=[], xmin=[], xmax=[],
              color='red', zorder=1, label = 'EDM rate change [cm/day]')
ax0.axvspan(xmin=datetime.datetime.strptime('2021-05-28','%Y-%m-%d'),xmax=datetime.datetime.strptime('2021-06-04','%Y-%m-%d'),facecolor='grey',alpha=0.5,)
ax0.annotate('no data', (datetime.datetime.strptime('2021-05-29','%Y-%m-%d'),-10),xycoords='data',rotation=90)
ax0.legend(loc='lower left')


ax2.plot([],[],color='tab:orange', marker='s',linestyle='-', label='summit dome volume')
ax2.plot([],[],color='tab:orange', marker='x',linestyle='--', label='SW dome volume')
ax2.plot([],[],color='tab:blue', marker='s',linestyle='-', label='summit dome extrusion rate')
ax2.plot([],[],color='tab:blue', marker='x',linestyle='--', label='SW dome extrusion rate')

ax2.legend(loc='upper left')

ax0.axvline(x=datetime.datetime.strptime('2020-10-13','%Y-%m-%d'),linewidth=1.5,color='black',linestyle='--')
ax0.axvline(x=datetime.datetime.strptime('2021-01-04','%Y-%m-%d'),linewidth=1.5,color='black',linestyle='--')
ax0.axvline(x=datetime.datetime.strptime('2021-07-01','%Y-%m-%d'),linewidth=1.5,color='black',linestyle='--')
ax0.axvline(x=datetime.datetime.strptime('2021-08-14','%Y-%m-%d'),linewidth=1.5,color='black',linestyle='--')

ax2.axvline(x=datetime.datetime.strptime('2020-10-13','%Y-%m-%d'),linewidth=1.5,color='black',linestyle='--')
ax2.axvline(x=datetime.datetime.strptime('2021-01-04','%Y-%m-%d'),linewidth=1.5,color='black',linestyle='--')
ax2.axvline(x=datetime.datetime.strptime('2021-07-01','%Y-%m-%d'),linewidth=1.5,color='black',linestyle='--')
ax2.axvline(x=datetime.datetime.strptime('2021-08-14','%Y-%m-%d'),linewidth=1.5,color='black',linestyle='--')

print(np.shape(EQ_data))
print(np.shape(EDM_dates))
# ax3.hist(np.arange(len(EDM_dates[:-1])),bins=EDM_dates,weights=EQ_data,stacked=True,ec='black')



for i, (a,eq_t) in enumerate(zip([ax3,ax4,ax5,ax6,ax7,ax8,ax9,ax10],EQ_types)):
    a.bar(EDM_dates[:-1], EQ_data[:,i], width, bottom=bottom,align='edge',label=eq_t)
    # a.set_ylabel(f'Event count: {eq_t}')
    a.set_xlim(x_lim_dates)
    a.legend(loc=(0.05,0.1))
    a.axvspan(xmin=datetime.datetime.strptime('2021-05-28','%Y-%m-%d'),xmax=datetime.datetime.strptime('2021-06-04','%Y-%m-%d'),facecolor='grey',alpha=0.5)
    a.axvline(x=datetime.datetime.strptime('2020-10-13','%Y-%m-%d'),linewidth=1.5,color='black',linestyle='--')
    a.axvline(x=datetime.datetime.strptime('2021-01-04','%Y-%m-%d'),linewidth=1.5,color='black',linestyle='--')
    a.axvline(x=datetime.datetime.strptime('2021-07-01','%Y-%m-%d'),linewidth=1.5,color='black',linestyle='--')
    a.axvline(x=datetime.datetime.strptime('2021-08-14','%Y-%m-%d'),linewidth=1.5,color='black',linestyle='--')
    a.tick_params(left=True,
                bottom=True,
                top=True,
                labelbottom=False,
                labeltop=False)

ax10.tick_params(left=True,
                bottom=True,
                top=True,
                labelbottom=True,
                labeltop=False)
ax7.set_ylabel('            Event count [-]\n ')

# fig.savefig('')




plt.show()

(88, 4)
(88,)
2021-11-05 00:00:00
2021-11-12 00:00:00
nan
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
(87, 8)
(88,)


In [96]:
plt.close('all')